# GeoAI Engine — fields, landcover, canopy

End-to-end walkthrough of the optional GeoAI runners shipped in `terraflow.geoai_engine`. The runners wrap [`opengeos/geoai`](https://github.com/opengeos/geoai) behind the same fingerprint + manifest + caching contract used elsewhere in TerraFlow.

This notebook stubs the model bodies so it executes without `geoai-py` or `torch` installed; the orchestrator code paths (config validation, fingerprinting, cache hits, artifact writing) are exercised exactly as they would be in production.

**Prereqs for a real run**: `pip install "terraflow-agro[geoai]"`.

## 1. Synthetic raster + config

In [ ]:
import tempfile, textwrap, json
from pathlib import Path
import numpy as np
import rasterio
from rasterio.transform import from_origin

tmp = Path(tempfile.mkdtemp(prefix='geoai_demo_'))
raster_path = tmp / 'land_cover.tif'
data = (np.random.default_rng(42).integers(0, 5, size=(64, 64))).astype('uint8')
with rasterio.open(
    raster_path, 'w',
    driver='GTiff', height=64, width=64, count=1, dtype='uint8',
    crs='EPSG:4326',
    transform=from_origin(0.0, 1.0, 1/64, 1/64),
) as dst:
    dst.write(data, 1)

(tmp / 'climate.csv').write_text('station_id,lat,lon,mean_temp,total_rain\n0,0.5,0.5,20,400\n')
config_yaml = textwrap.dedent(f'''
    raster_path: "{raster_path}"
    climate_csv: "{tmp / 'climate.csv'}"
    output_dir: "{tmp / 'outputs'}"
    roi:
      type: "bbox"
      xmin: 0.0
      ymin: 0.0
      xmax: 1.0
      ymax: 1.0
    model_params:
      v_min: 0.0
      v_max: 1.0
      t_min: 0.0
      t_max: 40.0
      r_min: 0.0
      r_max: 300.0
      w_v: 0.4
      w_t: 0.3
      w_r: 0.3
    geoai:
      engine: fields
      chip_size: 64
      confidence_threshold: 0.5
      batch_size: 4
    ''').strip()
cfg_path = tmp / 'config.yml'
cfg_path.write_text(config_yaml)
print('Config written to', cfg_path)

## 2. Stub the heavy ML deps

Without the `[geoai]` extra, `_GEOAI_AVAILABLE` is `False` and runners raise an `ImportError` with the install hint. For the demo we patch that flag plus the engine body so the orchestrator can execute against the synthetic raster.

In [ ]:
from terraflow import geoai_engine

geoai_engine._GEOAI_AVAILABLE = True
geoai_engine._do_fields = lambda cfg, run_dir: (
    (run_dir / 'fields.geojson').write_text('{"type": "FeatureCollection", "features": []}'),
    (run_dir / 'field_stats.parquet').write_bytes(b''),
)
print('Stub installed; ready to run.')

## 3. Run `run_fields` and inspect the artifact set

In [ ]:
run_dir = geoai_engine.run_fields(cfg_path)
print('Run directory:', run_dir)
print('Artifacts:')
for p in sorted(run_dir.iterdir()):
    print(' -', p.name, f'({p.stat().st_size} bytes)')

manifest = json.loads((run_dir / 'geoai_manifest.json').read_text())
print('\nManifest engine:', manifest['engine'])
print('Fingerprint:', manifest['geoai_fingerprint'][:16], '...')
print('Model device:', manifest['model']['device'])

## 4. Cache hit — second call reuses the run

Calling the runner again with the same config returns the same directory and **does not re-invoke** the engine body, because the fingerprinted manifest already exists.

In [ ]:
call_count = {'n': 0}
def counting_stub(cfg, run_dir):
    call_count['n'] += 1
    (run_dir / 'fields.geojson').write_text('{}')
    (run_dir / 'field_stats.parquet').write_bytes(b'')

geoai_engine._do_fields = counting_stub
_ = geoai_engine.run_fields(cfg_path)
_ = geoai_engine.run_fields(cfg_path)
print('Engine body invocations across two runs:', call_count['n'], '(expected 0 — cache hit)')

## 5. Change a config value → new fingerprint, new run

Bumping `confidence_threshold` from 0.5 → 0.9 produces a different fingerprint and therefore a different cache directory.

In [ ]:
cfg_b_yaml = config_yaml.replace('confidence_threshold: 0.5', 'confidence_threshold: 0.9')
cfg_b_path = tmp / 'config_b.yml'
cfg_b_path.write_text(cfg_b_yaml)
run_dir_b = geoai_engine.run_fields(cfg_b_path)
print('Original run:', run_dir)
print('Threshold-bumped run:', run_dir_b)
print('Different directories?', run_dir != run_dir_b)

## See also

- [GeoAI guide](../geoai.md) — installation + CLI usage
- [ADR-007: GeoAI Engine Adapter](../architecture/adr-007-geoai-engine.md) — design rationale
- [Reproducibility](../reproducibility.md) — what the fingerprint covers